# ViDDAR — live demo (for the video)

One photo from your computer, one object name, and the pipeline we actually kept:

**text query → Grounding DINO boxes → SAM picks the mask it trusts most → AR panel → obstructed or not.**

This is the method that won in `viddar_cross_dataset_comparison3` (SAM-ranked boxes). Comparison v2 showed that better DINO queries were the real gain, and that fine-tuning the mask decoder on synthetic images did not help, so this demo does not fine-tune anything. It runs zero-shot SAM ViT-B with the v3 box choice.

Obstruction uses ViDDAR’s rule: the AR panel obstructs the object when it covers at least **25%** of the mask.

**Before you record**

1. Colab → **Runtime → Change runtime type → T4 GPU**.
2. Run **[1] Setup** once (about 2 minutes). Leave that output off-camera if you want.
3. Run **[2]** and pick a photo from your computer.
4. In **[3]**, set `OBJECT_NAME`, then run the cell. That is the shot.
5. To try another object on the same photo, change the name and run **[3]** again. To try another photo, run **[2]** again.


## [1] Setup

In [ ]:
import io, os, sys, subprocess

import cv2
import numpy as np
import torch
from PIL import Image

print("CUDA:", torch.cuda.is_available(), "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to a GPU")
assert torch.cuda.is_available(), "This demo needs a GPU. Runtime → Change runtime type → T4."

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

subprocess.run(['pip', 'install', '-q', 'transformers'], check=False)

if not os.path.exists('/content/segment-anything'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/facebookresearch/segment-anything.git',
                    '/content/segment-anything'], check=True)

SAM_CKPT = '/content/sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_CKPT):
    print("Downloading SAM ViT-B (~375 MB)...")
    subprocess.run(['wget', '-q',
                    'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
                    '-O', SAM_CKPT], check=True)

sys.path.insert(0, '/content/segment-anything')
from segment_anything import sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

print("Loading Grounding DINO...")
dino_processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny").to("cuda").eval()

print("Loading SAM...")
sam = sam_model_registry['vit_b'](checkpoint=SAM_CKPT).cuda().eval()
resize_transform = ResizeLongestSide(sam.image_encoder.img_size)

DINO_SYNONYMS = {
    'car': ['car', 'automobile', 'vehicle'],
    'truck': ['truck', 'lorry', 'vehicle'],
    'bus': ['bus', 'vehicle'],
    'motorcycle': ['motorcycle', 'motorbike', 'bike'],
    'tv': ['tv', 'television', 'tv screen', 'monitor'],
    'stop sign': ['stop sign', 'stop'],
    'fire hydrant': ['fire hydrant', 'hydrant'],
    'traffic light': ['traffic light', 'traffic lights', 'stoplight'],
    'knife': ['knife', 'kitchen knife'],
    'scissors': ['scissors', 'pair of scissors'],
    'toaster': ['toaster'],
    'vehicle': ['car', 'truck', 'bus', 'motorcycle', 'vehicle'],
    'cup': ['cup', 'mug', 'coffee cup'],
    'mug': ['mug', 'cup', 'coffee cup'],
    'bottle': ['bottle', 'water bottle'],
    'phone': ['phone', 'cell phone', 'mobile phone'],
    'laptop': ['laptop', 'notebook computer'],
    'person': ['person', 'human'],
    'dog': ['dog'],
    'cat': ['cat'],
    'chair': ['chair'],
    'book': ['book'],
}

ALPHA = 0.25  # ViDDAR obstruction threshold
TOP_K_BOXES = 8
PHOTO = None  # filled by section [2]
PHOTO_NAME = None

print("Ready. Run section [2] and upload a photo.")


## [2] Upload a photo

Run this cell and choose a file from your computer (jpg or png). A clear photo of one object works best on camera: a mug, a laptop, a bottle, a chair.


In [ ]:
def _open_rgb(raw_bytes):
    img = Image.open(io.BytesIO(raw_bytes)).convert('RGB')
    # keep the frame readable on a laptop recording
    img.thumbnail((1280, 1280))
    return np.array(img)


if IN_COLAB:
    print("Pick a photo from your computer...")
    uploaded = colab_files.upload()
    assert uploaded, "No file selected."
    PHOTO_NAME = next(iter(uploaded))
    PHOTO = _open_rgb(uploaded[PHOTO_NAME])
else:
    from pathlib import Path
    path = input("Path to a photo on this computer: ").strip().strip('"')
    PHOTO_NAME = Path(path).name
    PHOTO = _open_rgb(Path(path).read_bytes())

print(f"Loaded {PHOTO_NAME}  {PHOTO.shape[1]}×{PHOTO.shape[0]}")


## [3] Run it — this is the frame to record

Edit the three lines at the top, then run the cell.

- `OBJECT_NAME` is what you say out loud (“find the mug”).
- `OBSTRUCT` places the AR card on the object. Set it to `False` to park the card in a free corner and show a clear scene.


In [ ]:
OBJECT_NAME = "cup"     # the object to find
OBSTRUCT = True         # True: AR card covers the object. False: card stays out of the way.
SHOW_OTHER_BOXES = True # faint boxes = other DINO candidates SAM rejected

import matplotlib.pyplot as plt


def dino_detect_all(image_pil, query, box_threshold=0.20, text_threshold=0.15, max_per_prompt=5):
    prompts = list(DINO_SYNONYMS.get(query.lower(), [query]))
    if query.lower() not in {p.lower() for p in prompts}:
        prompts = [query] + prompts
    detections = []
    for prompt in prompts:
        inputs = dino_processor(images=image_pil, text=[[prompt.lower()]], return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = dino_model(**inputs)
        results = dino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids, threshold=box_threshold, text_threshold=text_threshold,
            target_sizes=[image_pil.size[::-1]])[0]
        if len(results["boxes"]) == 0:
            continue
        scores = results["scores"].cpu().numpy()
        boxes = results["boxes"].cpu().numpy()
        for i in np.argsort(-scores)[:max_per_prompt]:
            detections.append((boxes[i], float(scores[i]), prompt))
    kept = []
    for box, sc, pr in sorted(detections, key=lambda x: -x[1]):
        x1, y1, x2, y2 = box
        drop = False
        for b2, _, _ in kept:
            xx1, yy1 = max(x1, b2[0]), max(y1, b2[1])
            xx2, yy2 = min(x2, b2[2]), min(y2, b2[3])
            inter = max(0, xx2 - xx1) * max(0, yy2 - yy1)
            a1 = max(0, x2 - x1) * max(0, y2 - y1)
            a2 = max(0, b2[2] - b2[0]) * max(0, b2[3] - b2[1])
            if inter / (a1 + a2 - inter + 1e-6) > 0.85:
                drop = True
                break
        if not drop:
            kept.append((box, sc, pr))
    return kept


def embed_image(image_rgb):
    inp = resize_transform.apply_image(image_rgb)
    h, w = inp.shape[:2]
    x = torch.as_tensor(inp, device="cuda").permute(2, 0, 1).contiguous()[None]
    with torch.no_grad():
        emb = sam.image_encoder(sam.preprocess(x))
    return {"emb": emb, "input_size": (h, w), "original_size": tuple(image_rgb.shape[:2])}


def sam_masks_for_box(sample, box):
    box_t = resize_transform.apply_boxes(np.asarray(box, dtype=np.float32)[None, :], sample["original_size"])
    box_t = torch.as_tensor(box_t, dtype=torch.float, device="cuda")
    with torch.no_grad():
        sparse, dense = sam.prompt_encoder(points=None, boxes=box_t, masks=None)
        low_res, iou_pred = sam.mask_decoder(
            image_embeddings=sample["emb"],
            image_pe=sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse,
            dense_prompt_embeddings=dense,
            multimask_output=True,
        )
        up = sam.postprocess_masks(low_res, input_size=sample["input_size"],
                                   original_size=sample["original_size"])
    masks = (up[0].cpu().numpy() > 0).astype(np.uint8)
    scores = iou_pred[0].detach().cpu().numpy()
    return [(masks[i], float(scores[i])) for i in range(masks.shape[0])]


def select_sam_ranked(sample, detections, top_k=TOP_K_BOXES):
    """v3: among DINO boxes, keep the mask with the highest SAM-predicted IoU."""
    if not detections:
        return None
    best = None
    for box, dino_sc, prompt in sorted(detections, key=lambda t: -t[1])[:top_k]:
        for mask, sam_iou in sam_masks_for_box(sample, box):
            key = (sam_iou, dino_sc)
            if best is None or key > (best[0], best[1]):
                best = (sam_iou, dino_sc, box, mask, prompt)
    sam_iou, dino_sc, box, mask, prompt = best
    return {"box": box, "mask": mask, "sam_iou": sam_iou, "dino_score": dino_sc, "prompt": prompt}


def overlay_mask(image, mask, color, alpha=0.45):
    out = image.copy()
    painted = out.copy()
    painted[mask.astype(bool)] = color
    out = cv2.addWeighted(painted, alpha, out, 1 - alpha, 0)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, color, 2)
    return out


def draw_box(image, box, color, width=3):
    out = image.copy()
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    cv2.rectangle(out, (x1, y1), (x2, y2), color, width)
    return out


def place_ar_card(image, key_mask, obstruct):
    """Translucent AR card. On the object, or in a corner that misses it."""
    h, w = image.shape[:2]
    color = np.array([37, 99, 235], dtype=np.float32)  # blue UI card
    ar = image.copy()
    content = np.zeros((h, w), dtype=np.uint8)
    ys, xs = np.where(key_mask > 0)
    if len(xs) == 0:
        return ar, content, 0.0
    x0, x1, y0, y1 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())
    bw, bh = max(8, x1 - x0 + 1), max(8, y1 - y0 + 1)
    if obstruct:
        pw = int(np.clip(bw * 0.72, 48, w))
        ph = int(np.clip(bh * 0.55, 36, h))
        cx, cy = int(xs.mean()), int(ys.mean())
        px0 = int(np.clip(cx - pw // 2, 0, max(0, w - 1)))
        py0 = int(np.clip(cy - ph // 2, 0, max(0, h - 1)))
        px1, py1 = int(np.clip(px0 + pw, 0, w)), int(np.clip(py0 + ph, 0, h))
    else:
        pw = int(np.clip(w * 0.28, 48, w))
        ph = int(np.clip(h * 0.16, 36, h))
        px0, py0, px1, py1 = w - pw - 8, 8, w - 8, 8 + ph
        patch = np.zeros_like(key_mask)
        patch[py0:py1, px0:px1] = 1
        if (patch & (key_mask > 0)).sum() / (key_mask.sum() + 1e-6) >= 0.06:
            px0, py0, px1, py1 = 8, 8, 8 + pw, 8 + ph
    content[py0:py1, px0:px1] = 1
    panel = ar.copy()
    panel[py0:py1, px0:px1] = color
    ar = cv2.addWeighted(panel, 0.55, ar, 0.45, 0)
    cv2.rectangle(ar, (px0, py0), (px1 - 1, py1 - 1), (255, 255, 255), 2)
    cv2.putText(ar, "AR", (px0 + 10, min(py1 - 10, py0 + 28)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    ratio = float((content & (key_mask > 0)).sum() / (key_mask.sum() + 1e-6))
    return ar, content, ratio


assert PHOTO is not None, "Run section [2] and upload a photo first."

pil = Image.fromarray(PHOTO)
dets = dino_detect_all(pil, OBJECT_NAME)
sample = embed_image(PHOTO)
picked = select_sam_ranked(sample, dets)

fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.6))
for ax in axes:
    ax.axis("off")

if picked is None:
    axes[0].imshow(PHOTO)
    axes[0].set_title("Your photo", fontsize=13, pad=8)
    for ax in axes[1:]:
        ax.imshow(np.full_like(PHOTO, 245))
    axes[2].set_title(f'No "{OBJECT_NAME}" found — try another name', fontsize=13, color="#b91c1c")
    fig.suptitle("ViDDAR  ·  nothing to segment", fontsize=16, fontweight="bold", y=1.02)
else:
    box, mask = picked["box"], picked["mask"]
    photo_boxes = PHOTO.copy()
    if SHOW_OTHER_BOXES:
        for b, sc, _ in dets:
            if not np.allclose(b, box, atol=1.0):
                photo_boxes = draw_box(photo_boxes, b, (180, 180, 180), width=2)
    photo_boxes = draw_box(photo_boxes, box, (14, 165, 233), width=4)
    x1, y1 = int(box[0]), int(box[1])
    cv2.putText(photo_boxes, f'{picked["prompt"]}  {picked["dino_score"]:.2f}',
                (max(4, x1), max(22, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (14, 165, 233), 2, cv2.LINE_AA)

    masked = overlay_mask(PHOTO, mask, (249, 115, 22), alpha=0.5)
    ar, _, ratio = place_ar_card(PHOTO, mask, OBSTRUCT)
    ar = overlay_mask(ar, mask, (249, 115, 22), alpha=0.28)
    obstructed = ratio >= ALPHA

    panels = [
        (PHOTO, "1  Your photo"),
        (photo_boxes, "2  Grounding DINO"),
        (masked, "3  SAM mask"),
        (ar, "4  AR card on the scene"),
    ]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title, fontsize=13, pad=8)

    verdict = "OBSTRUCTED" if obstructed else "CLEAR"
    color = "#b91c1c" if obstructed else "#15803d"
    fig.suptitle(
        f'Find “{OBJECT_NAME}”     {verdict}     '
        f'AR covers {ratio:.0%} of the object   (threshold {ALPHA:.0%})',
        fontsize=15, fontweight="bold", color=color, y=1.04,
    )
    print(
        f'query="{OBJECT_NAME}"  prompt used="{picked["prompt"]}"  '
        f'DINO={picked["dino_score"]:.2f}  SAM confidence={picked["sam_iou"]:.2f}  '
        f'candidates={len(dets)}  overlap={ratio:.1%}  → {verdict}'
    )

plt.tight_layout()
plt.show()
